In [20]:
from fp2mp_eval.utils import read_json
import pandas as pd
results = read_json('data/eval.json')

In [21]:
results

{'problem': 'Ты градостроительный эксперт. Оцени, насколько сценарий соответствует целям социально-экономического развития.\nВерни только JSON без markdown в формате:\n{"score": 0.0, "reasoning": "краткое объяснение на русском"}\n\nПравила:\n- 0.0 означает, что сценарий противоречит целям.\n- 0.5 означает частичное соответствие.\n- 1.0 означает максимально сильное соответствие.\n- Используй только данные из сценария.\n- В reasoning укажи 1-3 причины оценки.\n\nЦели СЭР:\nЦель - минимизировать негативное воздействие на окружающую среду, сохраняя при этом баланс с экономическими и социальными факторами.\n\nСценарий:\n{\n  "target_id": 7,\n  "site_area": 330924.5933878487,\n  "land_use": "BUSINESS",\n  "land_value_after": 60613852774.26909,\n  "investor_npv": 1659104435.47,\n  "params_repaired": {\n    "footprint_area": 14466.432100154389,\n    "l": 8.494103764520537,\n    "mxi": 0.07248652073489906,\n    "residential": 0.09060815091862381,\n    "business": 0.24141349202551496,\n    "recr

In [22]:
from fp2mp_eval.models import Evaluation
from fp2mp_eval import FP2MPEval
import pingouin as pg

if isinstance(results, dict):
    results = [results]

for result in results:
    evaluations = [Evaluation(**e) for e in result['evaluations']]
    result['evaluations_df'] = FP2MPEval.evaluations_to_df(evaluations)
    result['evaluations_long_df'] = FP2MPEval.evaluations_to_long_df(evaluations)
    result['icc_df'] = pg.intraclass_corr(result['evaluations_long_df'], targets='indicator', raters='judge', ratings='score')
    result['icc'] = {}
    for _,icc_row in result['icc_df'].iterrows():
        icc_type = icc_row['Type']
        icc_value = icc_row['ICC']
        result['icc'][icc_type] = icc_value
    result['score'] = result['evaluations_df'].mean().mean()

## Intraclass correlation

In [23]:
data = []

for result in results:
    problem = result['problem']
    model = result['model']
    icc = result['icc']
    for icc_type, icc_value in icc.items():
        data.append({
            'problem': problem,
            'model': model,
            'icc_type': icc_type,
            'icc_value': icc_value
        })

data_df = pd.DataFrame(data)
data_df.head()

,problem,model,icc_type,icc_value
0,"Ты градостроительный эксперт. Оцени, насколько...",llama3,"ICC(1,1)",0.027778
1,"Ты градостроительный эксперт. Оцени, насколько...",llama3,"ICC(A,1)",0.107383
2,"Ты градостроительный эксперт. Оцени, насколько...",llama3,"ICC(C,1)",0.181818
3,"Ты градостроительный эксперт. Оцени, насколько...",llama3,"ICC(1,k)",0.125000
4,"Ты градостроительный эксперт. Оцени, насколько...",llama3,"ICC(A,k)",0.375587


In [24]:
data_df.groupby(['icc_type']).agg(
        mean=("icc_value", "mean"),
        std=("icc_value", "std"),
    ).transpose()

icc_type,"ICC(1,1)","ICC(1,k)","ICC(A,1)","ICC(A,k)","ICC(C,1)","ICC(C,k)"
mean,0.027778,0.125,0.107383,0.375587,0.181818,0.526316
std,NaN,NaN,NaN,NaN,NaN,NaN


## Validation via models

In [25]:
data = []

for result in results:
    problem = result['problem']
    model = result['model']
    evaluations_df = result['evaluations_df']
    for judge in evaluations_df.index:
        for indicator in evaluations_df.columns:
            data.append({
                'problem': problem,
                'model': model,
                'judge': judge,
                'indicator': indicator,
                'value': evaluations_df.loc[judge, indicator]
            })

data_df = pd.DataFrame(data)
data_df

,problem,model,judge,indicator,value
0,"Ты градостроительный эксперт. Оцени, насколько...",llama3,0,framing,4
1,"Ты градостроительный эксперт. Оцени, насколько...",llama3,0,decomposition,4
2,"Ты градостроительный эксперт. Оцени, насколько...",llama3,0,diversity,4
3,"Ты градостроительный эксперт. Оцени, насколько...",llama3,0,coherence,4
4,"Ты градостроительный эксперт. Оцени, насколько...",llama3,0,justification,3
5,"Ты градостроительный эксперт. Оцени, насколько...",llama3,0,uncertainty_handling,3
6,"Ты градостроительный эксперт. Оцени, насколько...",llama3,0,knowledge_integration,3
7,"Ты градостроительный эксперт. Оцени, насколько...",llama3,0,metacognition,3
8,"Ты градостроительный эксперт. Оцени, насколько...",llama3,1,framing,4
9,"Ты градостроительный эксперт. Оцени, насколько...",llama3,1,decomposition,4


In [26]:
data_df.groupby(['model', 'indicator']).agg(
        mean=("value", "mean"),
        std=("value", "std"),
    ).unstack(level=0).reorder_levels([1, 0], axis=1).sort_index(axis=1)

model                 llama3          
                        mean       std
indicator                             
coherence                4.0  0.000000
decomposition            4.0  0.000000
diversity                3.8  0.447214
framing                  4.0  0.000000
justification            3.6  0.547723
knowledge_integration    3.4  0.894427
metacognition            3.6  0.547723
uncertainty_handling     3.6  0.547723